# Variants of the training recipe of c-ResUNet-light on Green (FNC v2) — Kaggle

One run:
1. five trainings of c-ResUNet-light (`--small_rf`, 100 epochs, no early stopping, seed 42), each with one change:
   `light_early_stop` (patience 15), `light_no_gamma`, `light_per_image` (per-image normalization), `light_wide`
   (32 base channels), `light_200ep` (200 epochs); after each, predictions on the validation images with `best.pt`;
2. test-time augmentation of the adopted network (`runs/green_v8/best.pt`) on the training, validation and test
   images → `runs/light_tta/`.

In this repository: `runs/cresunet_light_<variant>/` and `runs/cresunet_light/tta_check.json`.

Accelerator: **GPU T4 x2**. Inputs: the folders `cellseg-code` (code) and `fnc-green-data` (images and cleaned masks), found anywhere
under `/kaggle/input`. Short check with `PROBE = True`, then the full run with `PROBE = False` and
**Save Version → Save & Run All (Commit)**.

In [1]:
# 1) GPU available?
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: Settings → Accelerator → 'GPU T4 x2'"
print("GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128
GPU: Tesla T4


In [2]:
# 2) Find code and data under /kaggle/input, copy the code to /kaggle/working
import os
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def find_one(pattern):
    '''First path under INPUT matching the pattern (Kaggle can nest dataset folders in different ways).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} — did you add both datasets with 'Add Input'?"
    return hits[0]

CODE_SRC = find_one("models/c_res_unet_small_rf.py").parent.parent
IMAGES = find_one("green/trainval/images")
MASKS = find_one("cleaned_masks/trainval/Green/masks")
TEST_IMAGES = find_one("green/test/images")
TEST_MASKS = find_one("cleaned_masks/test/Green/masks")
print("code:  ", CODE_SRC)
print("images:", IMAGES, "→", len(list(IMAGES.glob("*.png"))), "png (expected 212)")

CODE = WORK / "code"   # /kaggle/input is read-only
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)
os.chdir(CODE)
assert (CODE / "runs/green_v8/best.pt").exists(), "old cellseg-code: upload the new version (with runs/green_v8/best.pt)"
print("working dir:", os.getcwd())

code:   /kaggle/input/datasets/alixram/cell-segmentation/cellseg-code
images: /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/green/trainval/images → 212 png (expected 212)
working dir: /kaggle/working/code


## 3) Parameters

- `PROBE = True`: short check (the widest variant, 1 epoch, few images; TTA on the validation images only);
  `PROBE = False`: full run.
- `VARIANTS`: variant name → options added after `BASE`, the recipe of c-ResUNet-light (when an option is repeated,
  the last value is used).

In [3]:
PROBE = False

BASE = ["--small_rf", "--epochs", "100", "--patience", "100"]          # recipe of the light network (green_v8)
VARIANTS = {
    "light_early_stop": ["--patience", "15"],
    "light_no_gamma": ["--no_gamma"],
    "light_per_image": ["--norm", "per_image"],
    "light_wide": ["--base_ch", "32"],
    "light_200ep": ["--epochs", "200", "--patience", "200"],
}
TTA_SPLITS = ["train", "val", "test"]
SPLIT_JSON = "splits/green_split.json"
LIMITS = []
if PROBE:
    VARIANTS = {"probe_light_wide": ["--base_ch", "32"]}               # the widest network: checks the GPU memory
    TTA_SPLITS = ["val"]
    LIMITS = ["--epochs", "1", "--patience", "1", "--train_limit", "8", "--val_limit", "4"]
print(list(VARIANTS), "| TTA on", TTA_SPLITS)

['light_early_stop', 'light_no_gamma', 'light_per_image', 'light_wide', 'light_200ep'] | TTA on ['train', 'val', 'test']


## 4) Training and predictions on the validation images, variant by variant

Stops at the first error. The predictions use `best.pt` (best validation Dice).

In [4]:
import subprocess
import sys
import time

def run(cmd):
    '''Runs a command, streams its output, stops on error.'''
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, f"failed (exit code {p.returncode})"

t_start = time.time()
for name, options in VARIANTS.items():
    t0 = time.time()
    out = WORK / "runs" / name
    run([sys.executable, "-m", "train.training", "--images_dir", IMAGES, "--masks_dir", MASKS, "--split_json", SPLIT_JSON,
         "--out_dir", out, "--num_workers", 4, *BASE, *options, *LIMITS])
    run([sys.executable, "-m", "evaluation.predict", "--ckpt", out / "best.pt", "--images_dir", IMAGES, "--masks_dir", MASKS,
         "--split_json", SPLIT_JSON, "--subset", "val", "--out", out / "pred_val.npz"])
    print(f"=== {name} done in {(time.time() - t0) / 60:.1f} min (total {(time.time() - t_start) / 60:.1f} min)", flush=True)

$ /usr/bin/python3 -m train.training --images_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/green/trainval/images --masks_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/cleaned_masks/trainval/Green/masks --split_json splits/green_split.json --out_dir /kaggle/working/runs/light_early_stop --num_workers 4 --small_rf --epochs 100 --patience 100 --patience 15
dati: 169 train, 43 val, caricati in 43 s | mean 0.0496 std 0.0699 | device cuda, amp True
ep   1 | lr 1.00e-03 | train loss 0.5433 | val loss 0.6518 dice 0.2059 | 72 s *
ep   2 | lr 1.00e-03 | train loss 0.3862 | val loss 0.4194 dice 0.6909 | 33 s *
ep   3 | lr 9.99e-04 | train loss 0.3636 | val loss 0.4006 dice 0.7032 | 37 s *
ep   4 | lr 9.98e-04 | train loss 0.3458 | val loss 0.4114 dice 0.6920 | 34 s 
ep   5 | lr 9.96e-04 | train loss 0.3474 | val loss 0.4081 dice 0.6933 | 35 s 
ep   6 | lr 9.94e-04 | train loss 0.3188 | val loss 0.3864 dice 0.7214 | 35 s *
ep   7 | lr 9.91e-04 | train l

## 5) Test-time augmentation of the adopted network

Mean of the probability maps over 8 symmetries (4 rotations × mirror), with `analysis/predict_tta.py`
(`experiments/predict_tta.py` in this repository).

In [5]:
tta_out = WORK / "runs" / ("probe_light_tta" if PROBE else "light_tta")
for split in TTA_SPLITS:
    images, masks = (TEST_IMAGES, TEST_MASKS) if split == "test" else (IMAGES, MASKS)
    run([sys.executable, "-m", "analysis.predict_tta", "--ckpt", "runs/green_v8/best.pt", "--images_dir", images,
         "--masks_dir", masks, "--split_json", SPLIT_JSON, "--subset", split, "--out", tta_out / f"pred_{split}.npz"])
print(f"total {(time.time() - t_start) / 60:.1f} min")

$ /usr/bin/python3 -m analysis.predict_tta --ckpt runs/green_v8/best.pt --images_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/green/trainval/images --masks_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/cleaned_masks/trainval/Green/masks --split_json splits/green_split.json --subset train --out /kaggle/working/runs/light_tta/pred_train.npz
runs/green_v8/best.pt (epoch 78) | TTA, 8 symmetries | 169 images in 229 s on cuda | saved to /kaggle/working/runs/light_tta/pred_train.npz
$ /usr/bin/python3 -m analysis.predict_tta --ckpt runs/green_v8/best.pt --images_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/green/trainval/images --masks_dir /kaggle/input/datasets/alixram/cell-segmentation/fnc-green-data/cleaned_masks/trainval/Green/masks --split_json splits/green_split.json --subset val --out /kaggle/working/runs/light_tta/pred_val.npz
runs/green_v8/best.pt (epoch 78) | TTA, 8 symmetries | 43 images in 60 s on cuda | saved to 

In [6]:
# 6) Summary: one row per variant
import json
import numpy as np
import pandas as pd

val_names = json.load(open(SPLIT_JSON))["val"]
rows = []
for name in VARIANTS:
    out = WORK / "runs" / name
    log = pd.read_csv(out / "log.csv")
    cfg = json.load(open(out / "config.json"))
    pred = np.load(out / "pred_val.npz")
    assert cfg["small_rf"] and list(pred["names"]) == val_names
    rows.append({"variant": name, "epochs": len(log), "best_epoch": int(log.epoch[log.val_dice_global.idxmax()]),
                 "best_dice": log.val_dice_global.max(), "base_ch": cfg["base_ch"], "norm": cfg["norm"],
                 "no_gamma": cfg["no_gamma"], "patience": cfg["patience"], "min": log.time_s.sum() / 60})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("TTA files:", sorted(p.name for p in tta_out.iterdir()))

         variant  epochs  best_epoch  best_dice  base_ch      norm  no_gamma  patience      min
light_early_stop      52          37     0.7570       16    global     False        15  30.9706
  light_no_gamma     100          78     0.7578       16    global      True       100  58.6165
 light_per_image     100          56     0.7532       16 per_image     False       100  58.6960
      light_wide     100          81     0.7613       32    global     False       100 111.2256
     light_200ep     200          81     0.7596       16    global     False       200 117.0749
TTA files: ['pred_test.npz', 'pred_train.npz', 'pred_val.npz']
